# AAAIM Evaluation Test

This notebook tests both single model evaluation and batch evaluation of multiple models.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()  # defaults to .env in current directory

# Add the project root to the Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import AAAIM functions
from core import annotate_model, curate_model
from core.database_search import force_clear_chromadb, get_species_recommendations_rag
# from core.database_search import get_reaction_recommendations_rag
from utils.evaluation import (
    evaluate_single_model,
    evaluate_models_in_folder,
    print_evaluation_results,
    compare_results,
    process_saved_llm_responses
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)



## Configuration

In [2]:
# LLM configuration
# llm_model = "Llama-4-Maverick-17B-128E-Instruct-FP8"
llm_model = "Llama-3.3-70B-Instruct"
# llm_model = "meta-llama/llama-3.3-70b-instruct:free"
# llm_model = "meta-llama/llama-3.3-70b-instruct"
# llm_model = "gpt-4.1-nano"

# Evaluation parameters
max_entities_per_model = 10  # Limit entities per model for testing
num_models_to_test = 5  # Number of models to test in batch evaluation

# Entity and database configuration
entity_type = "reaction"
database = "kegg"

output_dir = "./results/"  # Output directory for results

### Test function

In [3]:
# Test data - typical enzymes with synonyms
species_ids = ["PGK"]
synonyms_dict = {
    "PGK": ["hexokinase", "phosphoglucokinase"]
}

print("Testing RAG-based entity linking...")
print("="*50)

try:
    # Test RAG approach
    rag_recommendations = get_species_recommendations_rag(
        species_ids=species_ids,
        synonyms_dict=synonyms_dict,
        model_type="default",
        database='kegg'
    )
    
    for rec in rag_recommendations:
        print(f"\nSpecies: {rec.id}")
        print(f"Synonyms: {rec.synonyms}")
        print(f"Candidates: {rec.candidates}")
        print(f"Names: {rec.candidate_names}")
        print(f"Match scores (similarity): {rec.match_score}")
        
except Exception as e:
    print(f"RAG search failed: {e}")

2025-09-15 15:48:17,496 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-09-15 15:48:17,529 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2025-09-15 15:48:17,529 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Testing RAG-based entity linking...
c:\Users\user\Documents\research\AAAIM\data\chroma_storage
None
default


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.93it/s]
2025-09-15 15:48:19,414 - ERROR - Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]


Species: PGK
Synonyms: ['hexokinase', 'phosphoglucokinase']
Candidates: ['R01057', 'R08639', 'R09084']
Names: ['D-ribose 1,5-phosphomutase', 'alpha-D-glucose 1,6-phosphomutase', 'ADP:D-fructose-6-phosphate 1-phosphotransferase']
Match scores (similarity): [0.264, 0.25, 0.236]


## Annotating a new model with no or few existing annotations

In [4]:
model_file = "glycolysis_part1.xml"
# Check if test model exists
if os.path.exists(model_file):
    print(f"✓ Test model found: {model_file}")
else:
    print(f"✗ Test model not found: {model_file}")

✓ Test model found: glycolysis_part1.xml


In [5]:
# Test with a single model
recommendations_df, metrics = annotate_model(
    model_file=model_file,
    llm_model=llm_model,
    method="rag",
    top_k=10,
    entity_type=entity_type,
    database=database
)

2025-09-15 15:17:54,902 - INFO - Starting annotation for model: glycolysis_part1.xml
2025-09-15 15:17:54,903 - INFO - Using LLM model: Llama-3.3-70B-Instruct
2025-09-15 15:17:54,903 - INFO - Using method: rag for database search
2025-09-15 15:17:54,903 - INFO - Entity type: reaction, Database: kegg
2025-09-15 15:17:54,903 - INFO - >>>Step 1: Getting reactions from model...<<<
2025-09-15 15:17:54,906 - INFO - Found 6 reactions in model
2025-09-15 15:17:54,906 - INFO - Found 0 entities with existing annotations
2025-09-15 15:17:54,906 - INFO - Annotate all 6 entities
2025-09-15 15:17:54,906 - INFO - >>>Step 2: Extracting model context...<<<
2025-09-15 15:17:54,918 - INFO - Extracted context for model: small_test
2025-09-15 15:17:54,918 - INFO - >>>Step 3: Querying LLM (Llama-3.3-70B-Instruct)...<<<
2025-09-15 15:17:57,038 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-15 15:17:57,042 - INFO - LLM response received in 2.10s
2025-09-1

## Prefiltering by reaction participants

In [5]:
import time
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
import logging
import numpy as np
import re
from collections import Counter
import itertools
from core.model_info import find_species_with_chebi_annotations, find_species_with_annotations_and_qualifiers, find_species_with_ncbigene_annotations, find_species_with_uniprot_annotations, find_reactions_with_kegg_annotations, extract_model_info, format_prompt, get_all_species_ids
from core.model_info import get_all_reaction_ids
from core.llm_interface import get_system_prompt, query_llm, parse_llm_response
from core.data_types import Recommendation
from core.database_search import get_chromadb_client, get_species_recommendations_direct

In [6]:
entity_type = 'chemical'

entities_to_evaluate = get_all_species_ids(model_file, entity_type)
prompt = format_prompt(model_file, entities_to_evaluate, entity_type)
try:
    # Get appropriate system prompt for entity type
    system_prompt = get_system_prompt(entity_type)
    result = query_llm(prompt, system_prompt, model=llm_model, entity_type=entity_type)
    
    if not result:
        logger.error("No response from LLM")
        pd.DataFrame(), {"error": "No response from LLM"}
    
except Exception as e:
    logger.error(f"LLM query failed: {e}")

# Parse LLM response
synonyms_dict, reason = parse_llm_response(result)

2025-09-15 15:48:26,678 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"


In [7]:
a = "c:\\Users\\user\\Documents\\research\\AAAIM\\data\\chroma_storage"
client, collection = get_chromadb_client(a, "kegg_reactions_default", "default")

In [58]:
list_of_lists = [synonyms_dict[sp] for sp in synonyms_dict]
syn_list = [sp for synonyms_dict in list_of_lists for sp in synonyms_dict]

In [10]:
from rapidfuzz import fuzz

In [197]:
def exact_word_match(query, text):
    return query in text.split()

In [280]:
threshold = 85

all_metadatas = collection.get(include=["metadatas"])
ids = all_metadatas["ids"]
metadatas = all_metadatas["metadatas"]


filtered_reactions_dict = {}
for species in synonyms_dict:
    filtered_ids = []
    for query_word in synonyms_dict[species]:
        if query_word in cofactors_to_ignore:
            for idx, meta in zip(ids, metadatas):
                participants = meta.get("participants", [])
                max_score = [exact_word_match(query_word, p.lower()) for p in participants.split(';')]
                if any(max_score):
                    filtered_ids.append(idx)
        else: 
            for idx, meta in zip(ids, metadatas):
                participants = meta.get("participants", [])
                # Compute max fuzzy ratio for this record
                max_score = max(fuzz.partial_ratio(query_word.lower(), p.lower()) for p in participants.split(';')) if participants else 0
                if max_score >= threshold:
                    filtered_ids.append(idx)
                    # print(f"{query_word}--{idx}--{max_score} append fuzzy")

    filtered_reactions_dict[species] = set(filtered_ids)

In [74]:
entity_type = 'reaction'

entities_to_evaluate = get_all_reaction_ids(model_file)
prompt = format_prompt(model_file, entities_to_evaluate, entity_type)
try:
    # Get appropriate system prompt for entity type
    system_prompt = get_system_prompt(entity_type)
    result = query_llm(prompt, system_prompt, model=llm_model, entity_type=entity_type)
    
    if not result:
        logger.error("No response from LLM")
        pd.DataFrame(), {"error": "No response from LLM"}
    
except Exception as e:
    logger.error(f"LLM query failed: {e}")

# Parse LLM response
reaction_synonyms_dict, reason = parse_llm_response(result)

2025-09-15 18:03:29,345 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"


In [105]:
def extract_classifications(raw_text, classification):

    parts = []
    buf = ""
    paren_level = 0  # Track nested parentheses

    i = 0
    while i < len(raw_text):
        c = raw_text[i]

        # Track parentheses
        if c == '(':
            paren_level += 1
        elif c == ')':
            paren_level -= 1

        # Split points: + outside parentheses or <=>
        if c == '+' and paren_level == 0:
            parts.append(buf.strip())
            buf = ""
        elif raw_text[i:i+3] == '<=>' and paren_level == 0:
            parts.append(buf.strip())
            buf = ""
            i += 2  # skip the next two chars of <=>
        elif raw_text[i:i+2] == '->' and paren_level == 0:
            parts.append(buf.strip())
            buf = ""
            i += 1  
        else:
            buf += c

        i += 1

    # Add remaining buffer
    if buf:
        parts.append(buf.strip())
    # parts = [p for p in parts if p]
    strip_dollars = [p.lstrip("$") for p in parts if p]
    clean_lines = [re.sub(r'^[\d\w\(\)\+\-]+?\s+', '', p.strip()) for p in strip_dollars]

    return "; ".join(set(clean_lines))

In [282]:
model_info = extract_model_info(model_file, entities_to_evaluate, entity_type)
reaction_definitions = [i.split(':')[1] for i in model_info['reactions']]
reaction_participants = [extract_classifications(i, 'definition') for i in reaction_definitions]

reaction_participants

['ADP; Glucose; ATP; G6P',
 'F6P; G6P',
 'F6P; ADP; ATP; F16BP',
 'G3P; F16BP',
 'G3P',
 'ADP; ATP']

In [283]:
cofactors_to_ignore = {
    "ADP",
    "ATP"
}

In [284]:
import itertools

for participants in reaction_participants[:1]:
    part_list = (participants.split('; '))
    part_combos = list(itertools.combinations(part_list, 2))

part_combos

[('ADP', 'Glucose'),
 ('ADP', 'ATP'),
 ('ADP', 'G6P'),
 ('Glucose', 'ATP'),
 ('Glucose', 'G6P'),
 ('ATP', 'G6P')]

In [285]:
edited_combos = []
for combo in part_combos:
    if (combo[0] not in cofactors_to_ignore) or (combo[1] not in cofactors_to_ignore):
        edited_combos.append(combo)
edited_combos

[('ADP', 'Glucose'),
 ('ADP', 'G6P'),
 ('Glucose', 'ATP'),
 ('Glucose', 'G6P'),
 ('ATP', 'G6P')]

In [286]:
filtered_ids = set()
for i in edited_combos:
    intersecting_reactions = filtered_reactions_dict.get(i[0]).intersection(filtered_reactions_dict.get(i[1]))
    filtered_ids = filtered_ids | (intersecting_reactions)

filtered_ids = list(filtered_ids)

In [288]:
# Now query Chroma only on these filtered IDs
results = collection.query(
    query_texts=['Hexokinase', 'Glucokinase', 'Glucose kinase'],
    n_results=5,s
    include=["metadatas", "distances"],
    where={"kegg_id": {"$in": filtered_ids}}
)
results

Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]


{'ids': [['R07261', 'R01723', 'R06079', 'R03548', 'R08021'],
  ['R02888', 'R04491', 'R05981', 'R12208', 'R05980'],
  ['R01233', 'R09085', 'R02189', 'R05164', 'R11530']],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'name': 'NDP-glucose:1,4-alpha-D-glucan 4-alpha-D-glucosyltransferase',
    'definition': 'NDP-glucose + Amylose <=> NDP + Amylose',
    'pathways': '',
    'participants': 'NDP; Amylose; NDP-glucose',
    'equation': 'C15541 + C00718 <=> C00454 + C00718',
    'orthology': '',
    'ec_number': '2.4.1.242',
    'kegg_id': 'R07261',
    'brite': 'Hexosyltransferases; Glycosyltransferases; reactions; Transferase reactions; reactions [br08203.html]'},
   {'ec_number': '',
    'name': '',
    'participants': 'Nicotinurate; AMP; Nicotinate; Diphosphate; ATP; Glycine',
    'definition': 'Nicotinate + Glycine + ATP <=> Nicotinurate + AMP + Diphosphate',
    'brite': '',
    'orthology': '',
    'kegg

In [277]:
results['ids']

[['R03415', 'R02721', 'R13205', 'R12785', 'R10935'],
 ['R13512', 'R13513', 'R13001', 'R12668', 'R05465'],
 ['R01059', 'R01067', 'R10050', 'R10049', 'R01641']]

In [287]:
if "R01786" in filtered_ids:
    print(True)

True


In [ ]:
[r for r in reactions if r["kegg_id"] == "R00010"]

In [ ]:
recommendations_df

In [ ]:
metrics # luna

In [ ]:
metrics # janis

## Curate a model with existing annotations

Evaluation of a single model with existing annotations. 
Will only look at the species with existing annotations.

In [ ]:
model_file = "test_models/BIOMD0000000190.xml"
# Check if test model exists
if os.path.exists(model_file):
    print(f"✓ Test model found: {model_file}")
else:
    print(f"✗ Test model not found: {model_file}")

In [ ]:
# Test with a single model
recommendations_df, metrics = curate_model(
    model_file=model_file,
    llm_model=llm_model,
    method="rag",
    max_entities=max_entities_per_model,
    entity_type=entity_type,
    database=database
)

In [ ]:
metrics

In [ ]:
# janis
recommendations_df[recommendations_df['update_annotation'] != 'ignore']

In [ ]:
# luna
recommendations_df

## Test 1: Single Model Evaluation

Evaluation of a single model with existing annotations.

In [ ]:
# Test using utils evaluation function
model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000001046.xml"
result_df = evaluate_single_model(
    model_file=model_file,
    llm_model=llm_model,
    method = 'rag',
    top_k = 3,
    # max_entities=max_entities_per_model,
    entity_type=entity_type,
    database=database,
    save_llm_results=False,
    verbose=True
)

In [ ]:
result_df

## Test 2: Batch Model Evaluation

Test the evaluation of multiple models in a directory.

In [ ]:
model_dir = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels"
# model_dir = "test_models"
# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} XML files")
    # print(f"  - Will test first {min(num_models_to_test, len(model_files))} models")

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    method="rag",
    top_k=3,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-3_top3_general_prompt.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    method="rag",
    top_k=3,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-4_top3_general_prompt.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-3.3-70B-Instruct",
    method="rag",
    top_k=10,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-3_top10_prompt_adjusted.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    method="rag",
    top_k=1,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-4_top1_prompt_adjusted.csv",
    start_at=1,
    verbose=False
)

## Test 3: Evaluating previous LLM sysnonyms

In [ ]:
results_df = process_saved_llm_responses(response_folder = '/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/Llama-3.3-70B-instruct-Meta/chemical_prompt_adjusted', 
                               model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels', 
                               prev_results_csv = 'results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv', 
                               method = "direct",
                               output_dir = './results/', 
                               output_file = 'biomd_kegg_rag_meta-llama_top1_prompt_adjusted.csv',
                               top_k = 1,
                               verbose = False)

In [ ]:
results_df = process_saved_llm_responses(response_folder = '/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/llama-4-maverick-17b-128e-instruct-fp8/chemical_prompt_adjusted', 
                               model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels', 
                               prev_results_csv = 'results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv', 
                               method = "rag",
                               output_dir = './results/', 
                               output_file = 'biomd_kegg_rag_llama-4_top3_prompt_adjusted_rerun.csv',
                               top_k = 3,
                               verbose = False)

## Statistics

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top3_general_prompt.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top10_prompt_adjusted.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-3_top3_prompt_adjusted.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv")

In [ ]:
compare_results(
    'results/biomd_kegg_rag_meta-llama_top1_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top5_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top10_prompt_adjusted.csv',
    'results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv',
    'results/biomd_kegg_rag_llama-4_top10_prompt_adjusted.csv',
    'results/biomd_kegg_direct_llama-4_top3_prompt_adjusted.csv'
)

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv")

In [ ]:
df1 = pd.read_csv('results/biomd_kegg_rag_llama-4_top3_repeat2.csv')
df2 = pd.read_csv('results/biomd_kegg_rag_llama-4_top3_repeat2_rerun.csv')

# Merge on model and species_id, suffixing accuracy columns
merged = pd.merge(
    df1[['model', 'species_id', 'accuracy']],
    df2[['model', 'species_id', 'accuracy']],
    on=['model', 'species_id'],
    suffixes=('_1', '_2')
)

# Find rows where accuracy differs (including NaN vs value)
diffs = merged[
    (merged['accuracy_1'] != merged['accuracy_2']) |
    (merged['accuracy_1'].isnull() != merged['accuracy_2'].isnull())
]

# Print the differing model/species_id pairs
for _, row in diffs.iterrows():
    print(f"model: {row['model']}, species_id: {row['species_id']}, accuracy_1: {row['accuracy_1']}, accuracy_2: {row['accuracy_2']}")

In [ ]:
compare_results('results/biomd_kegg_rag_llama-4_top3.csv','results/biomd_kegg_rag_llama-4_top3_rerun.csv', 'results/biomd_kegg_rag_llama-4_top3_repeat2.csv', 'results/biomd_kegg_rag_llama-4_top3_repeat2_rerun.csv')

In [ ]:
recommendations_df[recommendations_df['id']=='Nb']

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_llama-4_top3.csv') 

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_meta-llama_default.csv')

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_gpt-4.1-nano_default.csv')

In [ ]:
gpt_df_filtered.to_csv('results/biomd_kegg_rag_gpt-4.1-nano_default_filtered.csv', index=False)

In [ ]:
prev_df = pd.read_csv('results/biomd_kegg_direct_llama_plain_nosymbols.csv')
prev_models = set(prev_df['model'].unique())
new_df = pd.read_csv('results/biomd_kegg_rag_meta-llama_top3.csv')
new_models = set(new_df['model'].unique())
new_models = new_models - prev_models
new_models

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
results_llama = pd.read_csv('results/biomd_kegg_rag_meta-llama_default.csv')
# Group models by number of species and calculate accuracy statistics
model_species_count = results_llama.groupby('model').size().reset_index(name='species_count')
model_accuracy = results_llama.groupby('model')['accuracy'].mean().reset_index()

# Merge the two dataframes
model_stats = pd.merge(model_species_count, model_accuracy, on='model')

# Create bins for species count to make the box plot more readable
bins = [0, 5, 10, 20, 50, 100, 1000]
labels = ['1-5', '6-10', '11-20', '21-50', '51-100', '100+']
model_stats['species_count_bin'] = pd.cut(model_stats['species_count'], bins=bins, labels=labels)

# Count number of models in each bin
bin_counts = model_stats['species_count_bin'].value_counts().reindex(labels, fill_value=0)

# Create the box plot
plt.figure(figsize=(6, 5))
ax = sns.boxplot(x='species_count_bin', y='accuracy', data=model_stats, medianprops={"color": "yellow", "linewidth": 1})

# Add counts above each box
for i, label in enumerate(labels):
    count = bin_counts[label]
    ax.text(i, 1.01, f'n={count}', ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')

plt.title('Model accuracy by number of species in a model')
plt.xlabel('Number of species in Model')
plt.ylabel('Average model accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
# plt.savefig(os.path.join(output_dir, 'accuracy_by_species_count.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# fix display name handling
ref_df = pd.read_csv('/Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv')
df = pd.read_csv('/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_meta-llama_default.csv')

# Replace 'display_name' in df with 'species_name' from ref_df using 'model' and 'species_id' as keys
df = df.merge(ref_df[['model', 'species_id', 'species_name']], on=['model', 'species_id'], how='left', suffixes=('', '_ref'))
df['display_name'] = df['species_name']
df = df.drop(columns=['species_name'])
df.to_csv('/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_meta-llama_default_display_names.csv', index=False)

# LLM synergy

In [ ]:
from utils.evaluation import evaluate_llm_synergy

# Compare multiple LLM results
synergy_df = evaluate_llm_synergy(
    'results/biomd_kegg_rag_meta-llama_top3.csv',
    'results/biomd_kegg_rag_gpt-4.1-nano_default.csv',
    'results/biomd_kegg_rag_llama-4_top3.csv',
    output_file='llm_synergy_analysis.csv'
)

# Save filtered results

In [ ]:
print_evaluation_results("/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_llama-3_top10_prompt_adjusted.csv", ref_results_csv=None)

In [ ]:
results_xlsx = "/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_llama-3_top10_prompt_adjusted_review.xlsx"
ref_results_csv = "/Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv"
df = pd.read_excel(results_xlsx)

# Filter by reference results
ref_df = pd.read_csv(ref_results_csv)
ref_pairs = set(zip(ref_df['model'], ref_df['species_id']))
mask = df.apply(lambda row: (row['model'], row['species_id']) in ref_pairs, axis=1)
df = df[mask]

print(f"Filtered results to {len(df)} entries that exist in reference: {ref_results_csv}")

In [ ]:
df.to_excel('results/biomd_kegg_rag_llama-3_top10_prompt_adjusted_is+isVersionOf_review.xlsx', index=False)